# 03 · Preprocesado común: oraciones, tokens y raíces

Tercer notebook del preprocesado (E2). Parte de los cuatro ficheros `*_limpio.jsonl` de los notebooks 01 y 02 y les aplica a todos **la misma función de preprocesado**:

1. Normalización Unicode (NFC).
2. División en oraciones.
3. Tokenización (`tokens`, para embeddings).
4. Raíces sin stopwords (`raices_bow`, para BoW / TF-IDF).

El resultado son los `*_procesado.jsonl`, con el formato de `ESQUEMA_DATOS.md`.

## 0. Preparación

In [36]:
import json
import unicodedata
from pathlib import Path

import pandas as pd
from nltk.tokenize import sent_tokenize

# El notebook se ejecuta desde la carpeta E2/, por eso las rutas empiezan por "..".
CARPETA_DATOS = Path("../data/processed")
FICHEROS = ["fallacyes_proto", "fallacyes_espont", "fallacyes_political", "congreso"]

corpus = {
    nombre: pd.read_json(CARPETA_DATOS / f"{nombre}_limpio.jsonl", lines=True, dtype=False)
    for nombre in FICHEROS
}
for nombre, tabla in corpus.items():
    print(f"{nombre}: {len(tabla)} textos")

fallacyes_proto: 3319 textos
fallacyes_espont: 1840 textos
fallacyes_political: 1951 textos
congreso: 2468 textos


## 1. Normalización Unicode (NFC)

Una letra con tilde se puede guardar de dos formas: como un solo carácter (`é`) o como la letra más una tilde suelta (`e` + `´`). Se ven igual, pero para Python son cadenas distintas, y el Word2Vec las trataría como dos palabras. NFC lo deja todo en la primera forma. **No quita las tildes.**

In [37]:
compuesta = "país"  # "i" + tilde suelta
print(compuesta, "==", "país", "→", compuesta == "país")
print("con NFC →", unicodedata.normalize("NFC", compuesta) == "país")

for nombre, tabla in corpus.items():
    cambian = tabla["texto"].map(lambda texto: unicodedata.normalize("NFC", texto) != texto)
    print(f"{nombre}: textos que cambian con NFC: {cambian.sum()}")

país == país → False
con NFC → True
fallacyes_proto: textos que cambian con NFC: 0
fallacyes_espont: textos que cambian con NFC: 0
fallacyes_political: textos que cambian con NFC: 0
congreso: textos que cambian con NFC: 0


En nuestros datos no cambia ningún texto. Se aplica igualmente: es una red de seguridad para textos nuevos (copiados de un PDF o escritos en un Mac) y es una línea de código.

## 2. División en oraciones: qué falla con `sent_tokenize`

`sent_tokenize` de NLTK usa el modelo Punkt para español. Punkt no corta en todos los puntos: tiene una lista de abreviaturas tras las que no corta. Se mira dónde se equivoca en nuestros datos.

In [38]:
pruebas = [
    "La Sra. García dijo que no. El Sr. López tampoco.",
    "Lo dijo el Prof. Martínez en clase.",
    "Vive en St. Louis desde hace años.",
    "Lo aprobó EE. UU. y luego Europa.",
    "Según el art. 155 de la Constitución.",
    "Vinieron todos: Juan, Ana, etc. Pero nadie habló.",
]
for prueba in pruebas:
    print(sent_tokenize(prueba, language="spanish"))

['La Sra.', 'García dijo que no.', 'El Sr. López tampoco.']
['Lo dijo el Prof.', 'Martínez en clase.']
['Vive en St.', 'Louis desde hace años.']
['Lo aprobó EE.', 'UU. y luego Europa.']
['Según el art.', '155 de la Constitución.']
['Vinieron todos: Juan, Ana, etc.', 'Pero nadie habló.']


In [39]:
# Cuántas veces sale cada abreviatura en cada corpus
ABREVIATURAS_A_BUSCAR = {
    "Sra.": r"\bSra\.", "Prof.": r"\bProf\.", "Dr. / Dra.": r"\bDra?\.", "St.": r"\bSt\.",
    "art.": r"\barts?\.", "núm.": r"\bnúm\.", "cap.": r"\bcap\.", "vol.": r"\bvol\.",
    "EE. UU.": r"EE\. ?UU\.", "etc.": r"\betc\.",
}
pd.DataFrame({
    nombre: {abreviatura: tabla["texto"].str.count(patron).sum() for abreviatura, patron in ABREVIATURAS_A_BUSCAR.items()}
    for nombre, tabla in corpus.items()
})

,fallacyes_proto,fallacyes_espont,fallacyes_political,congreso
Sra.,24,0,0,0
Prof.,8,0,0,0
Dr. / Dra.,12,0,0,0
St.,6,0,0,0
art.,0,0,1,0
núm.,0,0,0,1
cap.,0,0,0,4
vol.,0,0,0,3
EE. UU.,1,2,2,0
etc.,2,18,3,6


### 2.1 Oraciones muy largas

El texto de Political viene de una transcripción automática (WhisperX), que a veces puntúa poco. Se mira cuánto miden las oraciones de cada corpus.

In [40]:
oraciones_de_prueba = {
    nombre: tabla["texto"].map(lambda texto: sent_tokenize(texto, language="spanish")).explode()
    for nombre, tabla in corpus.items()
}
pd.DataFrame({
    nombre: {
        "oraciones": len(oraciones),
        "palabras (mediana)": oraciones.str.split().str.len().median(),
        "palabras (percentil 95)": oraciones.str.split().str.len().quantile(0.95),
        "palabras (máximo)": oraciones.str.split().str.len().max(),
        "más de 80 palabras": (oraciones.str.split().str.len() > 80).sum(),
    }
    for nombre, oraciones in oraciones_de_prueba.items()
})

,fallacyes_proto,fallacyes_espont,fallacyes_political,congreso
oraciones,5859.0,2745.0,4290.00,68449.0
palabras (mediana),13.0,16.0,16.00,20.0
palabras (percentil 95),30.0,38.0,44.55,61.0
palabras (máximo),90.0,83.0,191.00,252.0
más de 80 palabras,1.0,1.0,11.00,1125.0


## 3. División en oraciones

**Decisión:** se usa Punkt (el modelo de `sent_tokenize`) añadiéndole las abreviaturas que le faltan, **solo las que salen en nuestros datos** (sección 2): *Sra., Dra., Prof., St., EE., art., núm., cap., vol.* Punkt guarda sus abreviaturas en minúscula y sin el punto final. Añadirlas es la forma habitual de adaptar Punkt a un dominio.

- *EE.* basta para *EE. UU.*: *UU.* ya estaba en la lista.
- *etc.* seguido de mayúscula se deja cortar: casi siempre la frase termina ahí (*"Juan, Ana, etc. Pero nadie habló."*).
- **Las oraciones largas no se parten** (sección 2.1): son frases largas de verdad, no errores, y la más larga (252 palabras) cabe en un transformer.

Se crea un tokenizador propio en vez de modificar el que usa `sent_tokenize` por dentro, para que el cambio quede a la vista.

In [42]:
from nltk.tokenize import PunktTokenizer

ABREVIATURAS_QUE_FALTAN = {"sra", "dra", "prof", "st", "ee", "art", "núm", "cap", "vol"}

tokenizador_de_oraciones = PunktTokenizer("spanish")
# _params es un atributo interno de NLTK, pero es la forma habitual de añadir abreviaturas a Punkt
tokenizador_de_oraciones._params.abbrev_types.update(ABREVIATURAS_QUE_FALTAN)


def dividir_en_oraciones(texto):
    return tokenizador_de_oraciones.tokenize(texto)


# Los casos de la sección 2, más alguno en el que sí hay que cortar
for prueba in pruebas + ["Lo aprobó EE. UU. Luego Europa.", "Lo dice el núm. 3. Y el cap. 2 también."]:
    print(dividir_en_oraciones(prueba))

['La Sra. García dijo que no.', 'El Sr. López tampoco.']
['Lo dijo el Prof. Martínez en clase.']
['Vive en St. Louis desde hace años.']
['Lo aprobó EE. UU. y luego Europa.']
['Según el art. 155 de la Constitución.']
['Vinieron todos: Juan, Ana, etc.', 'Pero nadie habló.']
['Lo aprobó EE. UU.', 'Luego Europa.']
['Lo dice el núm. 3.', 'Y el cap. 2 también.']


In [43]:
# Cuántas oraciones salen antes y después del cambio
oraciones_nuevas = {nombre: tabla["texto"].map(dividir_en_oraciones) for nombre, tabla in corpus.items()}
pd.DataFrame({
    nombre: {
        "oraciones antes": len(oraciones_de_prueba[nombre]),
        "oraciones ahora": oraciones_nuevas[nombre].str.len().sum(),
    }
    for nombre in corpus
})

,fallacyes_proto,fallacyes_espont,fallacyes_political,congreso
oraciones antes,5859,2745,4290,68449
oraciones ahora,5819,2745,4289,68441


In [44]:
# Ejemplos de las prototípicas en los que ha cambiado la división
proto = corpus["fallacyes_proto"]["texto"]
antes = proto.map(lambda texto: sent_tokenize(texto, language="spanish"))
ahora = oraciones_nuevas["fallacyes_proto"]
for texto_antes, texto_ahora in list(zip(antes[antes != ahora], ahora[antes != ahora]))[:4]:
    print("antes:", texto_antes)
    print("ahora:", texto_ahora, "\n")

antes: ['La Sra.', 'Miller dio clases a tu hermano mayor, que era el payaso de la clase.', 'Decide ser estricta contigo desde el primer día de clase porque cree que tú también serás un payaso.']
ahora: ['La Sra. Miller dio clases a tu hermano mayor, que era el payaso de la clase.', 'Decide ser estricta contigo desde el primer día de clase porque cree que tú también serás un payaso.'] 

antes: ['La Sra.', 'Miller dará clases a ti y puede tener sus propias percepciones y expectativas sobre tu comportamiento en clase.', 'Esto no necesariamente significa que te tratará de manera diferente porque tu hermano mayor fue un payaso en su clase.']
ahora: ['La Sra. Miller dará clases a ti y puede tener sus propias percepciones y expectativas sobre tu comportamiento en clase.', 'Esto no necesariamente significa que te tratará de manera diferente porque tu hermano mayor fue un payaso en su clase.'] 

antes: ['Todos los profesores de escritura deben ser increíbles porque la Sra.', 'López lo es.']
aho

## 4. Tokenización: qué tokenizador usar

Se comparan los tres tokenizadores de NLTK vistos en clase:
- **Treebank** (`TreebankWordTokenizer`): reglas pensadas para el inglés del Penn Treebank.
- **wordpunct** (`wordpunct_tokenize`): separa cualquier secuencia de letras/números de cualquier secuencia de signos.
- **casual** (`casual_tokenize`): pensado para texto de redes sociales (emoticonos, menciones, palabras alargadas).

Primero se mira qué casos difíciles hay en nuestros datos y cuántas veces salen.

In [45]:
CASOS_DIFICILES = {
    "porcentaje (72%, 0,5 %)": r"\d\s?%",
    "decimal con coma (1,5)": r"\b\d+,\d+\b",
    "miles con punto (10.000)": r"\b\d{1,3}(?:\.\d{3})+\b",
    "palabra con guion (Pain-Away)": r"\b\w+-\w+\b",
    "apóstrofo (McDonald's, l’article)": r"\b\w+['’]\w+\b",
    "comillas «»": r"«",
    "comillas \"\"": r"\"",
    "signos de apertura ¿ ¡": r"[¿¡]",
    "puntos suspensivos": r"\.\.\.|…",
    "sigla con puntos (P.N.G., EE. UU.)": r"\b(?:[A-Z]{1,2}\.\s?){2,}",
    "emoticono (xD, :) )": r"(?<!\w)(?:[:;=]-?[)(DPp]|xD|XD)(?!\w)",
}
pd.DataFrame({
    nombre: {caso: tabla["texto"].str.count(patron).sum() for caso, patron in CASOS_DIFICILES.items()}
    for nombre, tabla in corpus.items()
})

,fallacyes_proto,fallacyes_espont,fallacyes_political,congreso
"porcentaje (72%, 0,5 %)",37,31,117,2031
"decimal con coma (1,5)",4,7,35,649
miles con punto (10.000),34,8,145,0
palabra con guion (Pain-Away),62,36,19,1256
"apóstrofo (McDonald's, l’article)",11,2,0,6542
comillas «»,0,3,0,640
"comillas """"",753,271,2,0
signos de apertura ¿ ¡,645,153,452,6294
puntos suspensivos,56,119,49,805
"sigla con puntos (P.N.G., EE. UU.)",13,2,2,5


In [46]:
from nltk.tokenize import TreebankWordTokenizer, casual_tokenize, wordpunct_tokenize

TOKENIZADORES = {
    "treebank": TreebankWordTokenizer().tokenize,
    "wordpunct": wordpunct_tokenize,
    "casual": casual_tokenize,
}

# Frases de nuestros corpus (algunas recortadas); las tres últimas son inventadas para probar siglas y emoticonos
frases_dificiles = [
    "¿Sabía que su tío fue encarcelado por malversar 1,5 millones de dólares?",       # proto
    "Preguntamos a más de 10.000 científicos y el 72% dijo que sí.",                  # proto
    "Pain-Away funcionó para mí, así que seguro que funciona para ti.",               # proto
    "Hamburguesas McDonald's, más de 99.000 millones servidas.",                      # proto
    "Es que no has atinado ni una sobre mi. xD Todo el mundo dice que soy muy majo",  # espont
    "Según la RAE, postula el «principio de igualdad de derechos».",                  # espont
    "¿Indefendible?, jajjajaaaaaaaaa.",                                               # espont
    "Derogar l’article…, l’apartat b), perdó, de l’article 6.2.",                     # congreso
    "Pero cuando miramos el crecimiento del P.N.G. este año...",                      # proto
    "El paro ha bajado un 0,5 % en EE. UU. desde 2023.",
    "El COVID-19 y la DANA del 29-O.",
    "Muy bien explicado :)",
]
for frase in frases_dificiles:
    print(frase)
    for nombre, tokenizar in TOKENIZADORES.items():
        print(f"   {nombre:<9}", tokenizar(frase))
    print()

¿Sabía que su tío fue encarcelado por malversar 1,5 millones de dólares?
   treebank  ['¿Sabía', 'que', 'su', 'tío', 'fue', 'encarcelado', 'por', 'malversar', '1,5', 'millones', 'de', 'dólares', '?']
   wordpunct ['¿', 'Sabía', 'que', 'su', 'tío', 'fue', 'encarcelado', 'por', 'malversar', '1', ',', '5', 'millones', 'de', 'dólares', '?']
   casual    ['¿', 'Sabía', 'que', 'su', 'tío', 'fue', 'encarcelado', 'por', 'malversar', '1,5', 'millones', 'de', 'dólares', '?']

Preguntamos a más de 10.000 científicos y el 72% dijo que sí.
   treebank  ['Preguntamos', 'a', 'más', 'de', '10.000', 'científicos', 'y', 'el', '72', '%', 'dijo', 'que', 'sí', '.']
   wordpunct ['Preguntamos', 'a', 'más', 'de', '10', '.', '000', 'científicos', 'y', 'el', '72', '%', 'dijo', 'que', 'sí', '.']
   casual    ['Preguntamos', 'a', 'más', 'de', '10.000', 'científicos', 'y', 'el', '72', '%', 'dijo', 'que', 'sí', '.']

Pain-Away funcionó para mí, así que seguro que funciona para ti.
   treebank  ['Pain-Away', 'funci

### 4.1 Resultado: se usa `casual_tokenize`

| | Treebank | wordpunct | casual |
|---|---|---|---|
| **¿ ¡ « »** | ❌ los deja pegados: `¿Sabía`, `«principio` | ✅ | ✅ |
| **Comillas ""** | ❌ las cambia por `` `` `` y `''` (modifica el texto) | ❌ las junta con otros signos: `".` | ✅ |
| **Cifras** (1,5 · 10.000 · 6.2) | ✅ | ❌ las rompe: `1` `,` `5` | ✅ |
| **Palabras con guion** (Pain-Away) | ✅ | ❌ | ✅ |
| **Apóstrofo recto** (McDonald's) | ❌ separa el `'s` del inglés | ❌ | ✅ |
| **Signos seguidos** (`?,` `»,` `),`) | ✅ | ❌ los junta en un token | ✅ |
| **Emoticonos** (`:)`, `xD`) | ❌ rompe `:)` | ✅ | ✅ |
| **Siglas con puntos** (P.N.G., EE. UU.) | ✅ | ❌ | ❌ las rompe: `P` `.` `N` `.` |
| **Apóstrofo curvo** (l’article, catalán) | ✅ | ❌ | ❌ `l` `’` `article` |
| **Sigla con número** (COVID-19) | ✅ | ❌ | ❌ `COVID` `-` `19` |

**Se usa `casual_tokenize`**: acierta en casi todo. wordpunct rompe las cifras (en el Congreso hay miles) y Treebank, hecho para el inglés, no separa ¿ ¡ « » y reescribe las comillas. Los tres fallos de casual (siglas con puntos, *COVID-19*, apóstrofo del catalán) son pocos casos. Se dejan sus opciones por defecto: conserva las mayúsculas (`preserve_case=True`) y no acorta las letras repetidas (`reduce_len=False`), porque los `tokens` se guardan sin modificar.

## 5. Raíces sin stopwords (`raices_bow`, para BoW / TF-IDF)

`raices_bow` es la versión de cada oración para BoW y TF-IDF:
- en minúsculas;
- sin stopwords;
- sin puntuación (salvo `?` y `!`, ver 5.1);
- con las cifras cambiadas por la marca `<num>` (ver 5.2);
- y con cada palabra reducida a su raíz con el stemmer Snowball de NLTK.

Así *país*, *pais* y *países* cuentan como la misma palabra, y el vocabulario es más pequeño.

**Solo para BoW / TF-IDF.** Los `tokens` (para embeddings) lo conservan todo: los preentrenados tienen vectores para las stopwords, el Word2Vec las necesita como contexto y los transformers usan el texto tal cual.

### 5.1 Qué stopwords se conservan

La lista de NLTK en español (313 palabras) quita palabras que pueden delatar una falacia. Para elegir cuáles se conservan, se mide en qué porcentaje de los textos aparece cada una, con falacia y sin falacia. **Solo con el `train`** de las prototípicas y las espontáneas: son los corpus con textos sin falacia, y mirar el test sería hacer trampa.

In [47]:
from nltk.corpus import stopwords

STOPWORDS_NLTK = stopwords.words("spanish")

train = pd.concat([corpus["fallacyes_proto"], corpus["fallacyes_espont"]])
train = train[train["split"] == "train"].copy()
train["clase"] = train["etiquetas"].map(lambda etiquetas: etiquetas[0] if etiquetas else "sin_falacia")
train["palabras"] = train["texto"].map(lambda texto: {token.lower() for token in casual_tokenize(texto)})


def porcentaje_de_textos_con(palabra, textos):
    return 100 * textos["palabras"].map(lambda palabras: palabra in palabras).mean()


# Candidatas: las 7 acordadas al principio y las que más destacan al medir todas las de NLTK
CANDIDATAS = ["o", "no", "porque", "todo", "todos", "ni", "nada",
              "tanto", "eres", "tú", "te", "tu", "qué", "nos", "nosotros", "contra", "tienes", "?", "!"]
CLASES = ["ad_hominem", "hombre_de_paja", "falso_dilema", "falsa_causalidad", "deduccion_invalida",
          "ad_populum", "generalizacion_apresurada", "distraccion"]

sin_falacia = train[train["clase"] == "sin_falacia"]
tabla = pd.DataFrame(index=CANDIDATAS)
tabla["% sin falacia"] = [porcentaje_de_textos_con(palabra, sin_falacia) for palabra in CANDIDATAS]
tabla["% con falacia"] = [porcentaje_de_textos_con(palabra, train[train["clase"] != "sin_falacia"]) for palabra in CANDIDATAS]
for clase in CLASES:
    # Cuántas veces más aparece en esa clase que en los textos sin falacia
    textos_de_la_clase = train[train["clase"] == clase]
    tabla[clase] = [(porcentaje_de_textos_con(palabra, textos_de_la_clase) + 0.5) / (porcentaje + 0.5)
                    for palabra, porcentaje in zip(CANDIDATAS, tabla["% sin falacia"])]
tabla.round(1)

,% sin falacia,% con falacia,ad_hominem,hombre_de_paja,falso_dilema,falsa_causalidad,deduccion_invalida,ad_populum,generalizacion_apresurada,distraccion
o,13.1,10.1,0.5,0.9,3.5,0.2,0.8,0.5,0.3,0.3
no,57.4,38.0,0.9,0.9,0.6,0.3,0.7,0.4,0.6,0.8
porque,7.7,11.9,1.3,1.4,0.5,1.6,0.9,2.0,0.9,0.4
todo,3.9,7.9,1.6,2.9,0.8,0.3,1.3,6.9,1.4,1.3
todos,7.5,8.6,0.5,0.9,0.8,0.6,1.6,2.2,2.2,0.9
ni,2.0,2.8,3.1,0.7,1.0,0.8,0.5,0.7,1.4,1.5
nada,2.1,2.4,1.3,1.9,1.3,0.2,0.8,0.6,0.7,1.4
tanto,2.2,7.8,0.9,1.4,0.5,6.7,13.4,1.5,4.8,1.7
eres,0.4,2.3,9.4,2.9,4.6,0.5,2.2,1.8,0.9,0.5
tú,1.0,2.8,4.4,1.8,1.0,0.8,0.3,6.9,0.5,3.8


Las columnas de cada clase dicen cuántas veces más sale la palabra en esa clase que en los textos sin falacia: 1 = igual; más de 1 = más en la falacia; menos de 1 = más en los textos sin falacia. Se suma 0,5 arriba y abajo para no dividir por cero.

**Decisión:** se conservan 17 palabras de la lista de NLTK, más los signos `?` y `!`:

| Grupo | Palabras | Por qué |
|---|---|---|
| Acordadas al principio | *o, no, porque, todo, todos, ni, nada* | Disyunción (falso dilema), negación, causa y cuantificadores universales. *No* y *o* salen menos en las falacias (muchas parejas no falaces niegan: *"esto **no** significa…"*), pero eso también sirve para distinguir, y *o* se dispara en el falso dilema |
| Conclusión | *tanto* | *"por lo **tanto**"*: muy frecuente en la deducción inválida y la falsa causalidad |
| Segunda persona | *eres, tú, te, tu, tienes* | Ataque directo a la persona (ad hominem): *"**tú** **eres** un…"* |
| Preguntas retóricas | *qué* | Hombre de paja y ad hominem: *"¿**qué** quiere, que…?"* |
| Bandos | *nos, nosotros, contra* | Falso dilema y hombre de paja: *"o con **nosotros** o **contra**…"* |
| Signos | `?` `!` | `?` sale en el 4 % de los textos sin falacia y en una cuarta parte de los hombres de paja y las distracciones |

Se descartan otras palabras que también destacaban (*mi*, *esa*, *he*…) pero sin una explicación clara: podrían ser casualidad de estos textos. *Si*, *nunca*, *siempre*, *nadie* y *usted* no están en la lista de NLTK, así que se conservan sin hacer nada. Los signos de apertura ¿ ¡ se quitan: repiten la información de `?` y `!`.

Ojo: la medición no incluye Political (no tiene textos sin falacia), y parte de las diferencias pueden venir de cómo se escribieron a mano las parejas no falaces.

In [48]:
from nltk.stem.snowball import SnowballStemmer

PALABRAS_QUE_MARCAN_FALACIAS = {
    "o", "no", "porque", "todo", "todos", "ni", "nada",  # acordadas al principio
    "tanto", "eres", "tú", "te", "tu", "tienes", "qué", "nos", "nosotros", "contra",  # añadidas tras medirlo (5.1)
}
STOPWORDS = set(STOPWORDS_NLTK) - PALABRAS_QUE_MARCAN_FALACIAS
SIGNOS_QUE_SE_CONSERVAN = {"?", "!"}
MARCA_DE_CIFRA = "<num>"

stemmer = SnowballStemmer("spanish")


def es_palabra_o_numero(token):
    return any(caracter.isalnum() for caracter in token)


def es_cifra(token):
    """Tiene dígitos y ninguna letra: "2023", "0,5", "2.900". "G20" no es una cifra."""
    return any(caracter.isdigit() for caracter in token) and not any(caracter.isalpha() for caracter in token)


def raices_para_bow(tokens):
    """Minúsculas, sin stopwords ni puntuación (salvo ? y !), cifras cambiadas por <num>
    y cada palabra reducida a su raíz."""
    raices = []
    for token in tokens:
        token = token.lower()
        if token in SIGNOS_QUE_SE_CONSERVAN:
            raices.append(token)
        elif es_cifra(token):
            raices.append(MARCA_DE_CIFRA)
        elif es_palabra_o_numero(token) and token not in STOPWORDS:
            raices.append(stemmer.stem(token))
    return raices


print("Stopwords que se quitan:", len(STOPWORDS), "de", len(STOPWORDS_NLTK))
for oracion in ["¿Y tú eres el que habla de corrupción? ¡Por favor!",
                "O estás con nosotros o estás contra nosotros.",
                "Todos los países lo hacen, por lo tanto no puede estar mal.",
                "El paro ha bajado un 0,5 % desde 2023 en España."]:
    print(oracion, "\n   →", raices_para_bow(casual_tokenize(oracion)))

Stopwords que se quitan: 296 de 313
¿Y tú eres el que habla de corrupción? ¡Por favor! 
   → ['tu', 'eres', 'habl', 'corrupcion', '?', 'favor', '!']
O estás con nosotros o estás contra nosotros. 
   → ['o', 'nosotr', 'o', 'contr', 'nosotr']
Todos los países lo hacen, por lo tanto no puede estar mal. 
   → ['tod', 'pais', 'hac', 'tant', 'no', 'pued', 'mal']
El paro ha bajado un 0,5 % desde 2023 en España. 
   → ['par', 'baj', '<num>', '<num>', 'españ']


Dos efectos del stemmer que hay que conocer:
- **Quita las tildes**: *país* → `pais`, *está* → `esta`. Así se juntan las variantes mal escritas del corpus espontáneo (*pais*, *tambien*).
- **A veces junta palabras distintas**: *señorías* y *señor* dan las dos `señor`.

### 5.2 Cifras: la marca `<num>`

Todo token que tiene dígitos y ninguna letra se cambia por `<num>`. Para detectar falacias no importa la cifra concreta, sino que **hay** una cifra: los datos y las estadísticas son típicos de la falacia de credibilidad y de la generalización. Además, cada cifra distinta sería una "palabra" que sale una sola vez y solo engorda el vocabulario.

También evita un fallo de `casual_tokenize`, que solo entiende un separador dentro de un número: *2.900.000* sale como `2.900` `.` `000` (13 casos en proto y Political), y las cifras con espacio del Diario, *14 400*, como `14` `400` (1.116 en el Congreso). Con la marca da igual cómo las parta: quedan como `<num>`.

Los `tokens` para embeddings se dejan como salen de `casual_tokenize`: para los embeddings las cifras apenas importan y casi nunca están en su vocabulario.

## 6. La función de preprocesado

`preprocesar` junta todas las piezas anteriores. Recibe el `texto` limpio de un ejemplo (o de una intervención) y su `id`, y devuelve la lista `oraciones` del esquema:

1. NFC (sección 1).
2. División en oraciones con las abreviaturas añadidas (sección 3).
3. Para cada oración:
   - `tokens` con `casual_tokenize` (sección 4);
   - `raices_bow` (sección 5);
   - un `id`: el del texto + `_s` + el número de la oración.

**El número de oración lleva 3 cifras** (`_s000`, `_s001`…): hay intervenciones del Congreso con más de 100 oraciones (la más larga tiene 387).

In [49]:
def preprocesar(texto, id_texto):
    """Divide el texto en oraciones y prepara cada una para embeddings (tokens) y para BoW (raices_bow)."""
    texto = unicodedata.normalize("NFC", texto)
    oraciones = []
    for numero, oracion in enumerate(dividir_en_oraciones(texto)):
        tokens = casual_tokenize(oracion)
        oraciones.append({
            "id": f"{id_texto}_s{numero:03d}",
            "texto": oracion,
            "tokens": tokens,
            "raices_bow": raices_para_bow(tokens),
        })
    return oraciones


# Un ejemplo real del corpus espontáneo
ejemplo = corpus["fallacyes_espont"].iloc[0]
print(ejemplo["texto"], "\n")
for oracion in preprocesar(ejemplo["texto"], ejemplo["id"]):
    print(json.dumps(oracion, ensure_ascii=False))

Para mi lo que diga este tipejo tiene cero validez. 

{"id": "fes_0000_s000", "texto": "Para mi lo que diga este tipejo tiene cero validez.", "tokens": ["Para", "mi", "lo", "que", "diga", "este", "tipejo", "tiene", "cero", "validez", "."], "raices_bow": ["dig", "tipej", "cer", "validez"]}


### 6.1 Pruebas con frases difíciles

In [50]:
FRASES_DIFICILES = {
    "siglas": "El PSOE y el PP votaron con EE. UU. y la OTAN en contra.",
    "números": "El paro bajó un 0,5 % en 2023: de 2.900.000 a 2.700.000 parados.",
    "abreviatura": "La Sra. García lo dijo en el art. 155.",
    "cita con «»": "Me dijo: «No pienso votarte. Eres un traidor». Y se fue.",
    "cita con \"\"": 'Como dijo Einstein: "La imaginación es más importante que el conocimiento". Y tenía razón.',
    "preguntas y exclamaciones": "¿Usted cree que esto es serio? ¡Por favor! Que dimita.",
    "puntos suspensivos": "Lo dijo el presidente… Y nadie le creyó.",
    "sin puntuación (tipo Whisper)": "o gobernamos nosotros o gobierna la derecha y eso lo sabe todo el mundo",
    "vacío": "",
}
for caso, frase in FRASES_DIFICILES.items():
    print(f"--- {caso}: {frase!r}")
    for oracion in preprocesar(frase, "prueba"):
        print(f"   {oracion['id']} {oracion['texto']!r}")
        print(f"        tokens:     {oracion['tokens']}")
        print(f"        raices_bow: {oracion['raices_bow']}")

--- siglas: 'El PSOE y el PP votaron con EE. UU. y la OTAN en contra.'
   prueba_s000 'El PSOE y el PP votaron con EE. UU. y la OTAN en contra.'
        tokens:     ['El', 'PSOE', 'y', 'el', 'PP', 'votaron', 'con', 'EE', '.', 'UU', '.', 'y', 'la', 'OTAN', 'en', 'contra', '.']
        raices_bow: ['pso', 'pp', 'vot', 'ee', 'uu', 'otan', 'contr']
--- números: 'El paro bajó un 0,5 % en 2023: de 2.900.000 a 2.700.000 parados.'
   prueba_s000 'El paro bajó un 0,5 % en 2023: de 2.900.000 a 2.700.000 parados.'
        tokens:     ['El', 'paro', 'bajó', 'un', '0,5', '%', 'en', '2023', ':', 'de', '2.900', '.', '000', 'a', '2.700', '.', '000', 'parados', '.']
        raices_bow: ['par', 'baj', '<num>', '<num>', '<num>', '<num>', '<num>', '<num>', 'par']
--- abreviatura: 'La Sra. García lo dijo en el art. 155.'
   prueba_s000 'La Sra. García lo dijo en el art. 155.'
        tokens:     ['La', 'Sra', '.', 'García', 'lo', 'dijo', 'en', 'el', 'art', '.', '155', '.']
        raices_bow: ['sra', 'garc

**Lo que sale de las pruebas:**
- Siglas, abreviaturas, preguntas y exclamaciones: bien.
- **Cifras**: en `tokens`, *2.900.000* se parte (`2.900` `.` `000`), pero en `raices_bow` todo queda como `<num>` (sección 5.2).
- **Citas con un punto dentro**: *«No pienso votarte. Eres un traidor».* se parte en dos oraciones, y cada comilla queda en una. Es correcto (son dos oraciones); solo queda raro a la vista.
- **Puntos suspensivos `…` seguidos de mayúscula**: Punkt no corta (*"Lo dijo el presidente… Y nadie le creyó."* queda en una). En el Congreso, `…` casi siempre marca una interrupción en mitad de la frase, así que no cortar es lo más seguro.
- **Texto sin puntuación**: queda como una sola oración larga. Se acepta (ver sección 2.1).
- **Texto vacío**: devuelve una lista vacía, sin error.

## 7. Aplicar a los cuatro corpus y exportar los `*_procesado.jsonl`

Cada `*_procesado.jsonl` tiene lo mismo que su `*_limpio.jsonl` más el campo `oraciones`. Se genera siempre a partir del limpio y nunca se edita a mano: si cambia algo del preprocesado, se vuelve a ejecutar este notebook. Los ficheros se comprueban en la sección 8.

In [52]:
from tqdm import tqdm

procesados = {}
for nombre, tabla in corpus.items():
    procesado = tabla.copy()
    procesado["oraciones"] = [
        preprocesar(texto, id_texto)
        for texto, id_texto in tqdm(zip(tabla["texto"], tabla["id"]), total=len(tabla), desc=nombre)
    ]
    procesados[nombre] = procesado

congreso: 100%|██████████| 2468/2468 [00:27<00:00, 90.46it/s] 


In [53]:
def guardar_jsonl(tabla, nombre_fichero):
    """Guarda la tabla en JSONL (una línea por fila). Los valores vacíos (NaN) se guardan como null."""
    ruta = CARPETA_DATOS / nombre_fichero
    registros = tabla.astype(object).where(tabla.notna(), None).to_dict(orient="records")
    with open(ruta, "w", encoding="utf-8") as fichero:
        for registro in registros:
            fichero.write(json.dumps(registro, ensure_ascii=False) + "\n")
    print(f"Guardado {ruta.name}: {len(registros)} textos, {ruta.stat().st_size / 1e6:.0f} MB")


for nombre, procesado in procesados.items():
    guardar_jsonl(procesado, f"{nombre}_procesado.jsonl")

Guardado fallacyes_proto_procesado.jsonl: 3319 textos, 4 MB
Guardado fallacyes_espont_procesado.jsonl: 1840 textos, 3 MB
Guardado fallacyes_political_procesado.jsonl: 1951 textos, 5 MB
Guardado congreso_procesado.jsonl: 2468 textos, 60 MB


## 8. Validación

Se valida lo que de verdad se entrega: los `*_procesado.jsonl` leídos del disco, no las tablas en memoria. La función `una_fila_por_oracion` sirve también para trabajar con una fila por oración (por ejemplo, para entrenar el Word2Vec).

In [56]:
from collections import Counter

leidos = {
    nombre: pd.read_json(CARPETA_DATOS / f"{nombre}_procesado.jsonl", lines=True, dtype=False)
    for nombre in FICHEROS
}


def una_fila_por_oracion(tabla):
    columnas_del_texto = ["id", "split"] if "split" in tabla else ["id", "uso"]
    por_oracion = tabla[columnas_del_texto + ["oraciones"]].explode("oraciones", ignore_index=True)
    campos_de_la_oracion = pd.json_normalize(por_oracion["oraciones"].tolist())
    return por_oracion.drop(columns="oraciones").rename(columns={"id": "id_texto"}).join(campos_de_la_oracion)


oraciones_leidas = {nombre: una_fila_por_oracion(tabla) for nombre, tabla in leidos.items()}

### 8.1 Los ficheros están completos y coinciden con la función actual

Se comprueba que no se ha perdido nada al guardar y que cada texto coincide con lo que genera ahora `preprocesar`. Si se cambia una función y no se vuelven a ejecutar todas las celdas, el kernel puede seguir usando la versión antigua y exportar ficheros desfasados.

**Si sale algún texto distinto: *Restart* del kernel y *Run All*.**

In [ ]:
for nombre, tabla in leidos.items():
    oraciones = oraciones_leidas[nombre]
    distintos = sum(
        preprocesar(texto, id_texto) != oraciones_del_texto
        for texto, id_texto, oraciones_del_texto in zip(tabla["texto"], tabla["id"], tabla["oraciones"])
    )
    print(nombre)
    print("   textos:", len(tabla), "| columnas: las del limpio + oraciones:",
          list(tabla.columns) == list(corpus[nombre].columns) + ["oraciones"])
    print("   textos sin oraciones:", (tabla["oraciones"].str.len() == 0).sum(),
          "| oraciones sin tokens:", (oraciones["tokens"].str.len() == 0).sum(),
          "| IDs de oración repetidos:", oraciones["id"].duplicated().sum())
    print("   textos distintos de lo que genera la función:", distintos)

### 8.2 Estadísticas por corpus

- **Vocabulario:** número de `tokens` distintos (respetando mayúsculas y tildes) y de raíces distintas.
- **Raíces que salen una sola vez:** cuánto vocabulario aporta poco al BoW.

In [58]:
def estadisticas(oraciones):
    tokens = Counter(token for lista in oraciones["tokens"] for token in lista)
    raices = Counter(raiz for lista in oraciones["raices_bow"] for raiz in lista)
    return {
        "textos": oraciones["id_texto"].nunique(),
        "oraciones": len(oraciones),
        "oraciones por texto (media)": len(oraciones) / oraciones["id_texto"].nunique(),
        "tokens": sum(tokens.values()),
        "tokens por oración (media)": sum(tokens.values()) / len(oraciones),
        "tokens por oración (máximo)": oraciones["tokens"].str.len().max(),
        "vocabulario de tokens": len(tokens),
        "vocabulario de raíces": len(raices),
        "% raíces que salen una sola vez": 100 * sum(n == 1 for n in raices.values()) / len(raices),
        "% de raíces que son <num>": 100 * raices[MARCA_DE_CIFRA] / sum(raices.values()),
    }


pd.DataFrame({nombre: estadisticas(oraciones) for nombre, oraciones in oraciones_leidas.items()}).round(1)

,fallacyes_proto,fallacyes_espont,fallacyes_political,congreso
textos,3319.0,1840.0,1951.0,2468.0
oraciones,5819.0,2745.0,4289.0,68441.0
oraciones por texto (media),1.8,1.5,2.2,27.7
tokens,96434.0,54243.0,91241.0,1864193.0
tokens por oración (media),16.6,19.8,21.3,27.2
tokens por oración (máximo),101.0,91.0,195.0,275.0
vocabulario de tokens,9427.0,9768.0,8518.0,61120.0
vocabulario de raíces,4634.0,5078.0,4001.0,26970.0
% raíces que salen una sola vez,25.7,51.0,39.5,39.4
% de raíces que son <num>,0.9,0.9,1.8,1.4


### 8.3 Lo que verá cada modelo

**BoW / TF-IDF:** el vocabulario se aprende del `train`. Las raíces del `test` que no están en el `train` se pierden (el vectorizador las ignora).

**Word2Vec propio:** se entrena con las oraciones `general` del Congreso. Con `min_count=5` (el valor por defecto de gensim), solo tienen vector las palabras que salen al menos 5 veces. Se mira qué parte de los tokens de cada corpus de falacias tendría vector.

In [59]:
def raices_de(oraciones):
    return Counter(raiz for lista in oraciones["raices_bow"] for raiz in lista)


print("Raíces del test que no están en el vocabulario del train (BoW / TF-IDF):")
for nombre in ["fallacyes_proto", "fallacyes_espont", "fallacyes_political"]:
    oraciones = oraciones_leidas[nombre]
    vocabulario_train = set(raices_de(oraciones[oraciones["split"] == "train"]))
    raices_test = raices_de(oraciones[oraciones["split"] == "test"])
    fuera = sum(n for raiz, n in raices_test.items() if raiz not in vocabulario_train)
    print(f"   {nombre}: {100 * fuera / sum(raices_test.values()):.1f} % de las raíces del test")

congreso_general = oraciones_leidas["congreso"][oraciones_leidas["congreso"]["uso"] == "general"]
frecuencia_en_congreso = Counter(token for lista in congreso_general["tokens"] for token in lista)
con_vector = {token for token, n in frecuencia_en_congreso.items() if n >= 5}

print("\nTokens de cada corpus que tendrían vector en un Word2Vec entrenado con el Congreso (min_count=5):")
for nombre in ["fallacyes_proto", "fallacyes_espont", "fallacyes_political"]:
    tokens = [token for lista in oraciones_leidas[nombre]["tokens"] for token in lista]
    cubiertos = sum(token in con_vector for token in tokens)
    print(f"   {nombre}: {100 * cubiertos / len(tokens):.1f} %")

Raíces del test que no están en el vocabulario del train (BoW / TF-IDF):
   fallacyes_proto: 8.5 % de las raíces del test
   fallacyes_espont: 11.8 % de las raíces del test
   fallacyes_political: 8.7 % de las raíces del test

Tokens de cada corpus que tendrían vector en un Word2Vec entrenado con el Congreso (min_count=5):
   fallacyes_proto: 87.4 %
   fallacyes_espont: 88.7 %
   fallacyes_political: 95.8 %


In [60]:
# Las palabras sin vector más frecuentes en el corpus espontáneo
tokens_espont = Counter(token for lista in oraciones_leidas["fallacyes_espont"]["tokens"] for token in lista)
[(token, n) for token, n in tokens_espont.most_common() if token not in con_vector][:40]

[('"', 271),
 ('(', 129),
 ('mas', 40),
 ('sabes', 25),
 ("'", 25),
 ('sólo', 24),
 ('Te', 22),
 ('EEUU', 17),
 ('dices', 16),
 ('Mejor', 12),
 ('crees', 10),
 ('economistas', 10),
 ('haces', 10),
 ('paises', 9),
 ('vacunas', 9),
 ('debes', 8),
 ('cuñado', 8),
 ('vacuna', 8),
 ('menéame', 7),
 ('cerebro', 7),
 ('tenéis', 7),
 ('contigo', 7),
 ('puta', 7),
 ('comunismo', 7),
 ('quieras', 7),
 ('Tú', 7),
 ('pais', 7),
 ('cojones', 6),
 ('tuyo', 6),
 ('USA', 6),
 ('creencias', 6),
 ('dios', 6),
 ('NO', 6),
 ('dedos', 5),
 ('tío', 5),
 ('Tienes', 5),
 ('..', 5),
 ('ciencias', 5),
 ('estudiado', 5),
 ('razonamiento', 5)]

### 8.4 Revisión a mano

Dos textos al azar de cada corpus (semilla fija), oración a oración.

In [61]:
SEMILLA = 42

for nombre, tabla in leidos.items():
    cortos = tabla[tabla["texto"].str.split().str.len() < 80]  # para que se puedan leer; en el Congreso casi todas son largas
    for _, fila in cortos.sample(2, random_state=SEMILLA).iterrows():
        print(f"===== {fila['id']}  etiquetas: {fila['etiquetas']}")
        for oracion in fila["oraciones"]:
            print(f"   {oracion['texto']}")
            print(f"      tokens:     {oracion['tokens']}")
            print(f"      raices_bow: {oracion['raices_bow']}")
        print()

===== fpr_1898_n  etiquetas: []
   El matrimonio ha sido tradicionalmente entre un hombre y una mujer, pero esto no significa que el matrimonio gay no debería estar permitido.
      tokens:     ['El', 'matrimonio', 'ha', 'sido', 'tradicionalmente', 'entre', 'un', 'hombre', 'y', 'una', 'mujer', ',', 'pero', 'esto', 'no', 'significa', 'que', 'el', 'matrimonio', 'gay', 'no', 'debería', 'estar', 'permitido', '.']
      raices_bow: ['matrimoni', 'sid', 'tradicional', 'hombr', 'muj', 'no', 'signif', 'matrimoni', 'gay', 'no', 'deb', 'permit']
   La ley y la sociedad han evolucionado y han cambiado su forma de pensar sobre el matrimonio.
      tokens:     ['La', 'ley', 'y', 'la', 'sociedad', 'han', 'evolucionado', 'y', 'han', 'cambiado', 'su', 'forma', 'de', 'pensar', 'sobre', 'el', 'matrimonio', '.']
      raices_bow: ['ley', 'socied', 'evolucion', 'cambi', 'form', 'pens', 'matrimoni']

===== fpr_1798_n  etiquetas: []
   Chico , esos suenan como robots bastante terribles, pero eso no necesari

### 8.5 Casos raros

In [62]:
def tiene_letras_o_cifras(texto):
    return any(caracter.isalnum() for caracter in texto)


for nombre, oraciones in oraciones_leidas.items():
    sin_palabras = oraciones[~oraciones["texto"].map(tiene_letras_o_cifras)]
    sin_raices = oraciones[oraciones["raices_bow"].str.len() == 0]
    # Empieza en minúscula sin ser la primera oración del texto: posible corte de más
    en_minuscula = oraciones[oraciones["texto"].str.match(r"[a-záéíóúñ]") & ~oraciones["id"].str.endswith("_s000")]
    print(nombre)
    print(f"   oraciones sin palabras: {len(sin_palabras)}", sin_palabras["texto"].value_counts().head(5).to_dict())
    print(f"   oraciones sin raíces:   {len(sin_raices)}", sin_raices["texto"].value_counts().head(5).to_dict())
    print(f"   empiezan en minúscula (sin ser la primera): {len(en_minuscula)}")

fallacyes_proto
   oraciones sin palabras: 12 {'".': 7, '...': 2, ').': 1, '.': 1, '?': 1}
   oraciones sin raíces:   12 {'".': 7, '...': 2, ').': 1, '.': 1, 'Para mí sí.': 1}
   empiezan en minúscula (sin ser la primera): 15
fallacyes_espont
   oraciones sin palabras: 6 {'!': 5, '?': 1}
   oraciones sin raíces:   2 {'Sí.': 2}
   empiezan en minúscula (sin ser la primera): 21
fallacyes_political
   oraciones sin palabras: 1 {'.': 1}
   oraciones sin raíces:   4 {'.': 1, 'Es por ti.': 1, 'Suyo.': 1, 'Y por eso, estamos donde estamos.': 1}
   empiezan en minúscula (sin ser la primera): 127
congreso
   oraciones sin palabras: 0 {}
   oraciones sin raíces:   62 {'Sí.': 24, 'Más.': 7, 'Sí, sí.': 5, 'Pero hay más.': 2, 'En esas estamos.': 1}
   empiezan en minúscula (sin ser la primera): 29


In [63]:
# Oraciones que son solo signos: ¿de dónde salen?
proto = oraciones_leidas["fallacyes_proto"]
for posicion in proto.index[~proto["texto"].map(tiene_letras_o_cifras)][:3]:
    print(f"   anterior: ...{proto.loc[posicion - 1, 'texto'][-70:]!r}")
    print(f"   suelta:   {proto.loc[posicion, 'texto']!r}\n")

# Political: oraciones que empiezan en minúscula
political = oraciones_leidas["fallacyes_political"]
en_minuscula = political.index[political["texto"].str.match(r"[a-záéíóúñ]") & ~political["id"].str.endswith("_s000")]
for posicion in en_minuscula[:4]:
    print(f"   anterior: ...{political.loc[posicion - 1, 'texto'][-60:]!r}")
    print(f"   siguiente: {political.loc[posicion, 'texto'][:60]!r}...\n")

   anterior: ...'nal porque todos suenan como si alguien estrangulara a un gato (¿o no?'
   suelta:   ').'

   anterior: ...'una cosa lleva a otra y a otra y a otra.'
   suelta:   '.'

   anterior: ...'¿Asesinato?'
   suelta:   '...'

   anterior: ...'que decía que cuando la economía falla lo demás son cuentos.'
   siguiente: 'pues las cuentas y los cuentos que ha dejado usted en este p'...

   anterior: ...'ón de personas ocupadas más que cuando llegamos al gobierno.'
   siguiente: 'más de un millón.'...

   anterior: ...'ue se acaba de citar, y no estaría mal hacer la comparación.'
   siguiente: 'en relación con la libra, que ocurre lo mismo, es un 10% más'...

   anterior: ...'En 1990 todavía se creó una muy buena cantidad de empleo.'
   siguiente: 'fue creándose menos en el 91 y ha decrecido el empleo en el '...



In [64]:
# Congreso: raíces de una letra (¿de dónde salen?)
raices_congreso = raices_de(oraciones_leidas["congreso"])
[(raiz, n) for raiz, n in raices_congreso.most_common() if len(raiz) == 1]

[('o', 6288),
 ('i', 6176),
 ('?', 6135),
 ('d', 2930),
 ('l', 2524),
 ('!', 835),
 ('s', 774),
 ('e', 348),
 ('m', 206),
 ('n', 129),
 ('a', 129),
 ('u', 73),
 ('b', 35),
 ('t', 15),
 ('º', 14),
 ('x', 8),
 ('v', 5),
 ('à', 5),
 ('f', 4),
 ('ª', 4),
 ('c', 3),
 ('p', 3),
 ('r', 2),
 ('j', 1),
 ('z', 1)]

### 8.6 Conclusiones de la validación

**Los ficheros están bien:** coinciden con lo que genera la función actual, no hay textos sin oraciones ni oraciones sin tokens, y la revisión a mano de 8 textos no ha encontrado errores de división ni de tokenización.

**Estadísticas (8.2):**
- Los corpus de falacias son textos cortos: entre 1,5 y 2,2 oraciones por texto y entre 17 y 21 tokens por oración.
- El Congreso tiene 28 oraciones por intervención y 1,86 M de tokens en total.
- El espontáneo tiene el vocabulario más disperso: la mitad de sus raíces salen una sola vez (faltas de ortografía, lenguaje de foro). Las prototípicas, solo una cuarta parte.

**Lo que verá cada modelo (8.3):**
- **BoW / TF-IDF:** entre el 8,5 % y el 11,8 % de las raíces del test no están en el vocabulario del train y se pierden. Es normal con corpus tan pequeños; el espontáneo es el peor por su vocabulario disperso.
- **Word2Vec propio entrenado con el Congreso:** tendría vector para el 87–89 % de los tokens de proto y espont y el 96 % de Political. Las que faltan son:
  - **Formas de *tú***: *sabes, dices, crees, haces, contigo, tenéis*. En el Congreso se habla de *usted*, así que apenas salen. Es justo el lenguaje del ad hominem del espontáneo: un Word2Vec solo del Congreso lo representará mal. Se puede entrenar también con el `train` de los corpus de falacias, o usar un preentrenado.
  - **Grafías distintas**: *sólo* (el Diario escribe *solo*), *EEUU* (el Diario, *EE. UU.*), *mas* sin tilde.
  - **Signos que el Congreso no usa**: comillas `"` (el Diario usa « ») y paréntesis (se quitaron con las acotaciones).
  - Pasar todo a minúsculas solo gana un punto de cobertura (88,3 / 89,7 / 96,2 %).

**Casos raros (8.5):**

| Caso | Cuántos | Qué pasa | Qué se hace |
|---|---|---|---|
| Oraciones que son solo signos (`".`, `!`, `...`) | 19 (12 proto, 6 espont, 1 Political) | Punkt corta tras `?` o `!` dentro de unas comillas o paréntesis y deja el cierre suelto: *"…¿o no?"* / *").* | Pendiente de decidir: pegarlas a la oración anterior |
| Oraciones que empiezan en minúscula | 127 en Political (3 %) | WhisperX puso un punto en mitad de la frase: *"…cuando llegamos al gobierno." / "más de un millón."* | Se deja: es la puntuación de la transcripción y no se puede arreglar con fiabilidad |
| Raíces de una letra en el Congreso (*i, d, l, s, e*) | ~13.000 (0,7 % de los tokens) | Catalán: *i* ("y") y los apóstrofos *d’*, *l’* partidos | Se deja (otras lenguas, ver notebook 02). El Congreso no se usa para entrenar el BoW |
| Oraciones sin raíces | 80 en total | Todas sus palabras son stopwords: *"Sí."*, *"Más."*, *"Es por ti."* | Se deja: son respuestas cortas |